In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
print("ok", cv2.__version__, mp.__version__, np.__version__, pd.__version__)


In [ ]:

from pathlib import Path

# ----------------------------
# Config
# ----------------------------
VIDEO_PATH = "input.mp4"   
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

SAVE_ANNOTATED_VIDEO = True
MAX_FACES = 1                     # set >1 if you want multiple faces
MIN_DET_CONF = 0.6
MIN_TRK_CONF = 0.6

# MediaPipe face mesh landmark indices useful for mouth metrics:
# - Upper inner lip: 13
# - Lower inner lip: 14
# - Left mouth corner: 61
# - Right mouth corner: 291
UPPER_LIP = 13
LOWER_LIP = 14
MOUTH_LEFT = 61
MOUTH_RIGHT = 291

# ----------------------------
# Helpers
# ----------------------------
def norm_to_pixel(x, y, w, h):
    """Convert normalized landmark coords to pixel coords."""
    return int(x * w), int(y * h)

def euclidean(p1, p2):
    return float(np.linalg.norm(np.array(p1) - np.array(p2)))

# ----------------------------
# Video I/O
# ----------------------------
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise FileNotFoundError(f"Could not open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

writer = None
if SAVE_ANNOTATED_VIDEO:
    out_video_path = str(OUT_DIR / "annotated.mp4")
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(out_video_path, fourcc, fps, (W, H))

mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

landmark_rows = []
metric_rows = []

with mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=MAX_FACES,
    refine_landmarks=True,  # includes iris refinement; still yields 468+ landmarks
    min_detection_confidence=MIN_DET_CONF,
    min_tracking_confidence=MIN_TRK_CONF,
) as face_mesh:

    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        # MediaPipe expects RGB
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb)

        # Default: no face
        if not results.multi_face_landmarks:
            metric_rows.append({
                "frame": frame_idx,
                "time_sec": frame_idx / fps,
                "has_face": 0,
                "mouth_open_px": np.nan,
                "mouth_width_px": np.nan,
            })
            if writer is not None:
                writer.write(frame)
            frame_idx += 1
            continue

        # Use the first face (common for mukbang)
        face_landmarks = results.multi_face_landmarks[0].landmark

        # Save all landmarks
        for i, lm in enumerate(face_landmarks):
            landmark_rows.append({
                "frame": frame_idx,
                "time_sec": frame_idx / fps,
                "landmark": i,
                "x_norm": lm.x,
                "y_norm": lm.y,
                "z_norm": lm.z,
                "x_px": lm.x * W,
                "y_px": lm.y * H,
            })

        # Compute simple mouth metrics in pixels
        ul = face_landmarks[UPPER_LIP]
        ll = face_landmarks[LOWER_LIP]
        ml = face_landmarks[MOUTH_LEFT]
        mr = face_landmarks[MOUTH_RIGHT]

        ul_px = np.array([ul.x * W, ul.y * H])
        ll_px = np.array([ll.x * W, ll.y * H])
        ml_px = np.array([ml.x * W, ml.y * H])
        mr_px = np.array([mr.x * W, mr.y * H])

        mouth_open = euclidean(ul_px, ll_px)
        mouth_width = euclidean(ml_px, mr_px)

        metric_rows.append({
            "frame": frame_idx,
            "time_sec": frame_idx / fps,
            "has_face": 1,
            "mouth_open_px": mouth_open,
            "mouth_width_px": mouth_width,
        })

        # Optional: draw landmarks on frame
        if writer is not None:
            annotated = frame.copy()
            mp_drawing.draw_landmarks(
                image=annotated,
                landmark_list=results.multi_face_landmarks[0],
                connections=mp_face_mesh.FACEMESH_TESSELATION,
                landmark_drawing_spec=None,
                connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_tesselation_style(),
            )
            mp_drawing.draw_landmarks(
                image=annotated,
                landmark_list=results.multi_face_landmarks[0],
                connections=mp_face_mesh.FACEMESH_CONTOURS,
                landmark_drawing_spec=None,
                connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_contours_style(),
            )

            # Put a tiny overlay of mouth metrics
            cv2.putText(
                annotated,
                f"mouth_open={mouth_open:.1f}px mouth_width={mouth_width:.1f}px",
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255, 255, 255),
                2,
                cv2.LINE_AA,
            )
            writer.write(annotated)

        frame_idx += 1
        if frame_idx % 100 == 0:
            print(f"Processed {frame_idx}/{frame_count} frames...")

cap.release()
if writer is not None:
    writer.release()

# Save outputs
landmarks_df = pd.DataFrame(landmark_rows)
metrics_df = pd.DataFrame(metric_rows)

landmarks_path = OUT_DIR / "face_landmarks.csv"
metrics_path = OUT_DIR / "mouth_metrics.csv"

landmarks_df.to_csv(landmarks_path, index=False)
metrics_df.to_csv(metrics_path, index=False)

print("Done.")
print(f"Saved landmarks: {landmarks_path}")
print(f"Saved mouth metrics: {metrics_path}")
if SAVE_ANNOTATED_VIDEO:
    print(f"Saved annotated video: {OUT_DIR / 'annotated.mp4'}")


c:\Users\rlz20\anaconda3\Lib\site-packages\tensorflow\python\keras\engine\training_arrays_v1.py:37: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.4.1)
  from scipy.sparse import issparse  # pylint: disable=g-import-not-at-top

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\rlz20\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\rlz20\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\rlz20\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\rlz20\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\rlz20\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\rlz20\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\rlz20\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\rlz20\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\rlz20\anaconda3\Lib\site-pack

...


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\rlz20\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\rlz20\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\rlz20\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\rlz20\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\rlz20\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Users\rlz20\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\rlz20\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "c:\Users\rlz20\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: initialization failed